# March 18 CE/PDM Pre-Gate Benchmark

This notebook benchmarks the pre-events periodicity gate that now routes confident periodic candidates away from the stochastic `events.py` branch.

Use it to:
- run the exact `apply_pre_periodicity_gate` logic used by `malca detect`
- tune CE/PDM thresholds on the March 18 light curves
- inspect the `periodic`, `ambiguous`, and `non_periodic` buckets before hardening defaults


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from malca.config.config_pipeline import MAG_BINS as ALL_MAG_BINS, WORKERS as DEFAULT_WORKERS
from malca.manifest import build_manifest
from malca.periodic_events import run_periodic_events
from malca.periodicity_gate import apply_pre_periodicity_gate

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 6)


/home/calder/miniforge3/envs/malca/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Point INPUT_TABLE at an existing manifest/tag parquet with dat_path,
# or leave it as None and build a flat-directory manifest from FLAT_LC_DIR.
# Default is all configured MALCA magnitude bins.
INPUT_TABLE = None
FLAT_LC_DIR = Path("/home/calder/code/malca/output/runs/runs_march18_bundle_all/bundle_assets/lightcurves")
INDEX_FILE = None
MAG_BINS = list(ALL_MAG_BINS)
N_WORKERS = DEFAULT_WORKERS
RUN_PERIODIC_BRANCH = True

OUTPUT_DIR = Path("output/diagnostics/march18_periodicity_pregate")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

GATE_KWARGS = {
    "min_period": 0.2,
    "max_period": 100.0,
    "n_periods": 5000,
    "n_bootstrap": 128,
    "significance_level": 1e-3,
    "pdm_snr_threshold": 5.0,
    "pdm_theta_threshold": 0.6,
    "ce_snr_threshold": 5.0,
    "ce_entropy_threshold": 0.6,
    "min_points": 50,
    "agreement_rel_tol": 0.1,
    "scatter_ratio_max": 0.8,
    "strong_single_scatter_ratio_max": 0.7,
    "lag_phase_max": 0.25,
    "strong_single_sig": 1e-4,
    "workers": N_WORKERS,
}


In [3]:
if INPUT_TABLE is not None:
    input_path = Path(INPUT_TABLE)
    if input_path.suffix.lower() in {".parquet", ".pq"}:
        df_input = pd.read_parquet(input_path)
    else:
        df_input = pd.read_csv(input_path)
else:
    df_input = build_manifest(
        None,
        None,
        mag_bins=MAG_BINS,
        id_column="asas_sn_id",
        flat_lc_dir=FLAT_LC_DIR,
        index_file=INDEX_FILE,
        show_progress=True,
        n_workers=N_WORKERS,
    )
    df_input = df_input[df_input["dat_exists"]].reset_index(drop=True)

print(f"Loaded {len(df_input):,} candidates")
display(df_input.head())


FileNotFoundError: Flat light-curve directory not found: /path/to/march18/bundle_assets/lightcurves

In [ ]:
checkpoint_path = OUTPUT_DIR / "march18_periodicity_gate_checkpoint.parquet"
df_gate = apply_pre_periodicity_gate(
    df_input,
    path_col="dat_path" if "dat_path" in df_input.columns else "path",
    checkpoint_path=checkpoint_path,
    show_tqdm=True,
    **GATE_KWARGS,
)
output_file = OUTPUT_DIR / "march18_periodicity_gate.parquet"
df_gate.to_parquet(output_file, index=False)
print(f"Saved gate output to {output_file}")

label_counts = df_gate["pre_periodicity_label"].value_counts(dropna=False).rename_axis("label").to_frame("n")
display(label_counts)
display(
    df_gate.groupby("pre_periodicity_label")[[
        "pre_periodicity_score",
        "pre_periodicity_scatter_ratio",
        "pre_pdm_snr",
        "pre_ce_snr",
    ]].median(numeric_only=True)
)


In [ ]:
if RUN_PERIODIC_BRANCH:
    df_periodic_input = df_gate[df_gate["pre_periodic_flag"]].copy()
    periodic_output = OUTPUT_DIR / "march18_periodic_events.parquet"
    if df_periodic_input.empty:
        df_periodic_events = pd.DataFrame()
        print("No confident periodic candidates routed to the periodic branch")
    else:
        df_periodic_events = run_periodic_events(
            df_periodic_input,
            path_col="dat_path" if "dat_path" in df_periodic_input.columns else "path",
            period_col="pre_periodicity_selected_period",
            excluded_cameras_col="excluded_cameras" if "excluded_cameras" in df_periodic_input.columns else None,
            workers=N_WORKERS,
            show_tqdm=True,
        )
        df_periodic_events.to_parquet(periodic_output, index=False)
        print(f"Saved periodic branch output to {periodic_output}")
        display(
            df_periodic_events[[
                "source_id",
                "phase_period_days",
                "dip_significant",
                "phase_dip_depth_mag",
                "phase_dip_depth_snr",
                "phase_dip_support_cycles",
                "phase_profile_reason",
            ]].sort_values("phase_dip_depth_snr", ascending=False).head(25)
        )
else:
    print("RUN_PERIODIC_BRANCH is False; skipping periodic branch benchmark")


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

df_gate["pre_periodicity_label"].value_counts().plot.bar(ax=axes[0, 0], title="Gate Labels")
axes[0, 0].set_ylabel("count")

df_gate["pre_periodicity_score"].dropna().plot.hist(ax=axes[0, 1], bins=40, title="Pre-Periodicity Score")
axes[0, 1].set_xlabel("-log10(min bootstrap sig)")

df_gate["pre_periodicity_scatter_ratio"].dropna().plot.hist(ax=axes[1, 0], bins=40, title="Folded Scatter Ratio")
axes[1, 0].set_xlabel("folded/raw scatter")

axes[1, 1].scatter(df_gate["pre_pdm_snr"], df_gate["pre_ce_snr"], s=8, alpha=0.5)
axes[1, 1].set_title("PDM vs CE SNR")
axes[1, 1].set_xlabel("pre_pdm_snr")
axes[1, 1].set_ylabel("pre_ce_snr")

plt.tight_layout()
plt.show()


In [ ]:
cols = [
    "source_id",
    "mag_bin",
    "pre_periodicity_label",
    "pre_periodicity_method",
    "pre_periodicity_selected_period",
    "pre_periodicity_score",
    "pre_periodicity_scatter_ratio",
    "pre_pdm_snr",
    "pre_ce_snr",
    "pre_periodicity_reason",
]

print("Top confident periodic candidates")
display(df_gate[df_gate["pre_periodic_flag"]].sort_values("pre_periodicity_score", ascending=False)[cols].head(50))

print("Ambiguous candidates to inspect before hard thresholding")
display(df_gate[df_gate["pre_periodicity_label"] == "ambiguous"].sort_values("pre_periodicity_score", ascending=False)[cols].head(50))
